# TP 1 — Préparation et lecture globale de nanoGPT — CORRIGÉ

## Objectif général

Dans ce TP, tu prépares l'environnement Jupyter et tu comprends le rôle des fichiers principaux de nanoGPT.

L'idée importante : avant de comprendre un LLM, il faut être capable de se repérer dans le repo qui contient le modèle, l'entraînement, l'inférence et la préparation des données.


## Sources principales du parcours

Ces notebooks se basent principalement sur nanoGPT de Karpathy et sur la documentation officielle PyTorch/Jupyter.

- nanoGPT — README et quick start : https://github.com/karpathy/nanoGPT#quick-start  
  À lire pour comprendre le flux officiel : préparer Shakespeare, entraîner avec `train.py`, puis générer avec `sample.py`.
- nanoGPT — `model.py` : https://github.com/karpathy/nanoGPT/blob/master/model.py  
  À lire comme colonne vertébrale : `LayerNorm`, `CausalSelfAttention`, `MLP`, `Block`, `GPTConfig`, `GPT.forward`, `GPT.generate`.
- nanoGPT — `train.py` : https://github.com/karpathy/nanoGPT/blob/master/train.py  
  À lire pour la boucle d'entraînement : batch, forward, loss, backward, optimizer, checkpoint.
- nanoGPT — `sample.py` : https://github.com/karpathy/nanoGPT/blob/master/sample.py  
  À lire pour l'inférence : chargement du checkpoint, encodage du prompt, `model.generate`.
- Config Shakespeare caractère : https://github.com/karpathy/nanoGPT/blob/master/config/train_shakespeare_char.py  
  À lire pour comprendre les hyperparamètres pédagogiques : `block_size`, `n_layer`, `n_head`, `n_embd`, `dropout`, `max_iters`.
- PyTorch — Tensors : https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html  
  À lire pour comprendre pourquoi les données, poids et activations sont des tenseurs.
- PyTorch — `torch.nn.Embedding` : https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html  
  À lire pour comprendre la table qui transforme des indices de tokens en vecteurs.
- PyTorch — `torch.nn.LayerNorm` : https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html  
  À lire pour comprendre la normalisation utilisée dans chaque bloc.
- PyTorch — `torch.nn.GELU` : https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html  
  À lire pour comprendre l'activation du MLP.
- PyTorch — `torch.nn.CrossEntropyLoss` : https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html  
  À lire pour comprendre la loss de prédiction du prochain token.
- PyTorch — `torch.nn.functional.scaled_dot_product_attention` : https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html  
  À lire pour comprendre la version optimisée de l'attention Q/K/V.
- PyTorch — CUDA semantics : https://docs.pytorch.org/docs/stable/notes/cuda.html  
  À lire pour comprendre le placement CPU/GPU des tenseurs et modèles.
- Jupyter — Markdown cells : https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Working%20With%20Markdown%20Cells.html  
  À lire pour savoir structurer les réponses dans les cellules Markdown.


## Partie 1 — Préparer l'environnement

### Question 1.1 — Vérifier Python, PyTorch et CUDA

**Solution :**


In [ ]:
from pathlib import Path
import os, sys, subprocess, textwrap, json, math, time
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


### Explication pédagogique

PyTorch peut exécuter des calculs sur CPU ou sur GPU CUDA.  
`torch.cuda.is_available()` ne veut pas dire que ton code utilise déjà le GPU ; cela veut seulement dire que PyTorch voit un GPU compatible.

Dans un LLM, cette distinction devient centrale :
- le CPU orchestre souvent le programme Python ;
- le GPU accélère les gros calculs tensoriels, comme les multiplications matricielles de l'attention et du MLP ;
- les tenseurs doivent être placés sur le bon device (`cpu`, `cuda`, parfois `mps`) pour que les opérations soient cohérentes.

**Erreur fréquente :** croire qu'installer PyTorch suffit à utiliser le GPU. En réalité, il faut aussi déplacer le modèle et les tenseurs sur le device GPU.


## Partie 2 — Récupérer le code nanoGPT

### Question 1.2 — Cloner nanoGPT

**Solution :**


In [ ]:
from pathlib import Path
import subprocess, sys, os

repo = Path("nanoGPT")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/karpathy/nanoGPT.git"], check=True)
else:
    print("Le dossier nanoGPT existe déjà.")

print("Repo prêt :", repo.resolve())


### Explication pédagogique

Le repo nanoGPT est notre laboratoire.  
On ne va pas seulement lancer des commandes : on va lire le code source pour comprendre comment un GPT est construit.

Le README du repo donne un chemin pédagogique très adapté : préparer Shakespeare, entraîner un modèle caractère par caractère, puis générer du texte avec `sample.py`.


### Question 1.3 — Identifier les fichiers importants

**Solution :**


In [ ]:
from pathlib import Path

files = [
    "nanoGPT/model.py",
    "nanoGPT/train.py",
    "nanoGPT/sample.py",
    "nanoGPT/config/train_shakespeare_char.py",
    "nanoGPT/data/shakespeare_char/prepare.py",
]

for f in files:
    p = Path(f)
    print(f"{f:45s} ->", "OK" if p.exists() else "MANQUANT")


### Explication pédagogique

Tu dois retenir cette carte mentale :

- `model.py` : définit le réseau GPT.
- `train.py` : entraîne le réseau.
- `sample.py` : charge un modèle et génère du texte.
- `config/train_shakespeare_char.py` : fixe une configuration petite et pédagogique.
- `data/shakespeare_char/prepare.py` : transforme le texte brut en entiers.

C'est exactement le flux modèle → données → entraînement → checkpoint → inférence.


## Partie 3 — Lire le repo comme un ingénieur

### Question 1.4 — Relier fichier et responsabilité

**Solution :**

| Fichier | Rôle dans le TP |
|---|---|
| `model.py` | Décrit l'architecture GPT : embeddings, attention causale, MLP, blocs Transformer, forward pass et génération. |
| `train.py` | Contient la boucle d'entraînement : chargement de batches, forward, loss, backward, optimizer step, évaluation, checkpoint. |
| `sample.py` | Charge un checkpoint ou GPT-2, encode un prompt, appelle `model.generate`, puis décode les tokens générés. |
| `config/train_shakespeare_char.py` | Configure un mini-GPT caractère par caractère pour Shakespeare : taille du modèle, `block_size`, `batch_size`, `max_iters`, etc. |
| `data/shakespeare_char/prepare.py` | Télécharge/prépare Shakespeare et crée `train.bin`, `val.bin`, `meta.pkl`. |


### Explication pédagogique

Un bon ingénieur LLM ne lit pas un repo au hasard. Il identifie d'abord les responsabilités.

Ici, `model.py` est le cœur conceptuel. Les autres fichiers l'utilisent :
- `train.py` appelle `GPT.forward` avec des targets pour produire une loss ;
- `sample.py` appelle `GPT.generate` pour produire du texte ;
- la config donne les hyperparamètres ;
- la préparation de données fournit des entiers que le modèle peut apprendre à prédire.


## Partie 4 — Préparer Shakespeare

### Question 1.5 — Lancer la préparation des données

**Solution :**


In [ ]:
from pathlib import Path
import os, subprocess

repo = Path("nanoGPT")
assert repo.exists(), "Clone d'abord le repo nanoGPT."

# On exécute le script depuis le dossier du repo.
subprocess.run([sys.executable, "data/shakespeare_char/prepare.py"], cwd=repo, check=True)


### Explication pédagogique

Un LLM n'apprend pas directement depuis un fichier texte brut.  
Il apprend sur une séquence d'entiers.

Pour le dataset Shakespeare caractère, chaque caractère devient un identifiant entier. Le script crée :
- `train.bin` : données d'entraînement ;
- `val.bin` : données de validation ;
- `meta.pkl` : dictionnaires de conversion caractère ↔ identifiant.

Cette étape est le pont entre langage humain et calcul numérique.


### Question 1.6 — Vérifier les fichiers créés

**Solution :**


In [ ]:
from pathlib import Path

data_dir = Path("nanoGPT/data/shakespeare_char")
for name in ["train.bin", "val.bin", "meta.pkl"]:
    p = data_dir / name
    print(f"{name:10s} ->", "OK" if p.exists() else "MANQUANT", "|", p)


### Explication pédagogique

`train.py` lit directement ces fichiers binaires avec `np.memmap`.  
Cela évite de relire et retraiter le texte brut à chaque batch.

Ce point est important pour ton objectif AI Engineer : les performances ne dépendent pas seulement du modèle, mais aussi de la façon dont les données sont stockées, chargées et déplacées vers CPU/GPU.
